# Part 4 — Full pipeline, ablations, cost and figures (local GPU)
Sparse scene → upsampler → RGB assignment → PTv3 → predictions mapped to the **original dense** Area 5 points → metrics.

**Prerequisites:** Part 2 (trained `A_dense`, `B_sparse25`, `D_dense_aug`) and Part 3 (`full_x4`, folds, ablation upsamplers) are done. All results go to `workspace/results/*.json`; finished rows and upsampled scenes already written are skipped.
The training knobs are read from `workspace/pcseg_settings.json`, so row H automatically uses the same budget as A, B and D.

Rows evaluated here
| row | PTv3 trained on | PTv3 tested on | needs training? |
|---|---|---|---|
| E | dense | sparse + naive interpolation ×4 | no |
| F (optional) | dense | sparse + pretrained Grad-PU ×4 | no |
| G | dense | sparse + **our upsampler** ×4 | no |
| H | **our upsampled train scenes** (cross-fit) | sparse + our upsampler ×4 | yes |

Rows A, B, C, D come from Part 2. **Expect small differences or even negative results**: report them honestly, the question "when does upsampling help segmentation" is the contribution.

## 0. Setup, checks and Pointcept

In [ ]:
import sys
from pathlib import Path
PROJECT = Path.cwd().resolve()
if not (PROJECT / "pcseg").is_dir():          # notebook opened from the notebooks/ folder
    PROJECT = PROJECT.parent
sys.path.insert(0, str(PROJECT))

import json, os, shutil
import numpy as np
import torch
from pcseg import settings, run, env_setup, results
settings.ensure_dirs()
D, S25, S12 = str(settings.DENSE), str(settings.SP25), str(settings.SP12)

g = settings.gpu_info()
print("torch", torch.__version__, "| cuda", torch.version.cuda, "| gpu:", f"{g[0]} ({g[1]:.0f} GB)" if g else "NONE")
print("workspace:", settings.ROOT)
print("data     :", settings.DATA)
print(f"free disk: {shutil.disk_usage(settings.ROOT).free / 2**30:.0f} GB")

UP, INT = str(settings.UP25X4), str(settings.INTERP25X4)
CK, CODE = str(settings.UPSAMPLER), str(settings.PROJECT)
K = settings.load_knobs()
print('knobs:', K)

In [ ]:
assert torch.cuda.is_available(), "No GPU visible to PyTorch: see README, Troubleshooting"
from pcseg import data_check, experiments
assert data_check.check(), "fix the data problems listed above first"
env_setup.setup_all()                       # Pointcept checkout, patch, shims, import test (fast if already done)
import pandas as pd, matplotlib.pyplot as plt
from pcseg.common import load_room, room_dirs, write_json, CLASS_NAMES
assert (settings.EXP / 'A_dense' / 'model' / 'model_last.pth').exists(), 'A_dense checkpoint missing: run Part 2 section 6'
assert (settings.UPSAMPLER / 'full_x4' / 'last.pt').exists(), 'upsampler full_x4 missing: run Part 3 section 4'

## 1. Upsampled Area 5 scenes (naive interpolation and our model)

In [ ]:
run.py_module('pcseg.pu_infer', '--method', 'interp', '--sparse-root', S25, '--dense-root', D, '--dst', INT, '--areas', 'Area_5')
run.py_module('pcseg.pu_infer', '--method', 'model', '--ckpt', f'{CK}/full_x4/last.pt', '--sparse-root', S25, '--dense-root', D, '--dst', UP, '--areas', 'Area_5')

## 2. Rows E and G: the dense-trained PTv3 on upsampled sparse scenes (test only)

In [ ]:
rE = run.evaluate_run('A_dense', 'interp25x4', INT, 'E_dense_model_on_interp25x4')
rG = run.evaluate_run('A_dense', 'up25x4', UP, 'G_dense_model_on_upsampled25x4')
print(f"E (interpolation): {rE['mIoU']*100:.1f} | G (our upsampler): {rG['mIoU']*100:.1f}")

### Row F (optional): pretrained Grad-PU
Grad-PU needs its own CUDA ops and a pretrained checkpoint, so it is not automated here. To add it: run its inference on every sparse Area 5 patch and write the merged scene with
`pu_data.write_upsampled_dataset(dst, S25, ['Area_5'], make_fn, dense_root=D)` (`make_fn(coord, color)` must return the upsampled `(coord, color)`), then call
`run.evaluate_run('A_dense', 'gradpu25x4', dst, 'F_dense_model_on_gradpu25x4')`.

## 3. Row H: train PTv3 on upsampled training scenes (cross-fit) and test on upsampled Area 5
Training areas are upsampled with fold models that did **not** see them (Part 3 section 5). `K.stride > 1` (set it in `workspace/pcseg_settings.json`) upsamples every k-th room only (cheaper: if you use it, state it and note that H then sees fewer rooms than A/B/D).
This is the most expensive part: about 200 rooms to upsample plus one full training run. Check the free disk space first (printed below); the upsampled training scenes are large.

In [ ]:
print(f"free disk: {shutil.disk_usage(settings.ROOT).free / 2**30:.0f} GB | row-H room stride: {K.stride}")
for fold, areas in (('foldA', ['Area_4', 'Area_6']), ('foldB', ['Area_1', 'Area_2', 'Area_3'])):
    assert (settings.UPSAMPLER / f'{fold}_x4' / 'last.pt').exists(), f'{fold}_x4 missing: run Part 3 section 5'
    run.py_module('pcseg.pu_infer', '--method', 'model', '--ckpt', f'{CK}/{fold}_x4/last.pt', '--sparse-root', S25, '--dense-root', D,
                  '--dst', UP, '--areas', *areas, '--label-areas', *areas, '--room-stride', K.stride)

In [ ]:
cfgH = experiments.make_H(K)              # same batch / point_max / epoch / AMP as A, B, D (read from the shared settings)
run.train('H_up_trained', cfgH)
rH = run.evaluate_run('H_up_trained', 'up25x4', UP, 'H_trained_on_upsampled')
print(f"H: {rH['mIoU']*100:.1f} mIoU")

## 4. Main results table

In [ ]:
rows = results.collect()
main = [r for r in rows if r['name'][:2] in ('A_', 'B_', 'C_', 'D_', 'E_', 'F_', 'G_', 'H_')]
df = pd.DataFrame(main).set_index('name')
print(df[['mIoU', 'OA', 'mAcc']].round(2).to_markdown())
print()
print(df[CLASS_NAMES].round(1).to_markdown())
ax = df['mIoU'].plot.barh(figsize=(8, 0.5 * len(df) + 1.5), color='#4c72b0'); ax.invert_yaxis(); ax.set_xlabel('mIoU (%) on original dense Area 5')
plt.tight_layout(); plt.savefig(settings.FIGURES / 'main_results.png', dpi=140); plt.show()
(settings.RESULTS / 'main_results.md').write_text(df[['mIoU', 'OA', 'mAcc']].round(2).to_markdown() + '\n\n' + df[CLASS_NAMES].round(1).to_markdown())

## 5. Ablations (test-only, PTv3 = dense-trained `A_dense`)
Each variant: upsample sparse Area 5 with a different upsampler/setting, test with the same PTv3, score on dense points. Each takes upsampling time + one test run: run only what you need.
Notes: the three loss variants use the **development** upsamplers (trained on Areas 1–4), so compare them with each other, not with `G`.

In [ ]:
variants = {
    'abl_rgb_idw':          dict(ckpt='full_x4',        sparse=S25, ratio=4, extra=['--rgb-mode', 'idw']),
    'abl_include_input':    dict(ckpt='full_x4',        sparse=S25, ratio=4, extra=['--include-input']),
    'abl_x2':               dict(ckpt='full_x2',        sparse=S25, ratio=2, extra=[]),
    'abl_loss_cd_only':     dict(ckpt='abl_cd_only',    sparse=S25, ratio=4, extra=[]),
    'abl_loss_cd_rep':      dict(ckpt='abl_cd_rep',     sparse=S25, ratio=4, extra=[]),
    'abl_loss_cd_rep_uni':  dict(ckpt='abl_cd_rep_uni', sparse=S25, ratio=4, extra=[]),
}
if os.path.isdir(S12) and os.path.exists(f'{CK}/full_x8_sp12/last.pt'):
    variants['abl_sparse12_x8'] = dict(ckpt='full_x8_sp12', sparse=S12, ratio=8, extra=[])
    run.evaluate_run('A_dense', 'sparse12', S12, 'C_dense_model_on_sparse12')       # reference without upsampling at 12.5 %

for name, v in variants.items():
    if (settings.RESULTS / f'{name}.json').exists():
        print('already done:', name); continue
    if not os.path.exists(f"{CK}/{v['ckpt']}/last.pt"):
        print('skip', name, '(upsampler checkpoint missing)'); continue
    dst = str(settings.DATA / f's3dis_{name}')
    run.py_module('pcseg.pu_infer', '--method', 'model', '--ckpt', f"{CK}/{v['ckpt']}/last.pt", '--sparse-root', v['sparse'], '--dense-root', D,
                  '--dst', dst, '--areas', 'Area_5', '--up-ratio', v['ratio'], *v['extra'], tail=1)
    r = run.evaluate_run('A_dense', name, dst, name)
    print(f"{name}: {r['mIoU']*100:.1f} mIoU")
    shutil.rmtree(dst, ignore_errors=True)

## 6. Computation cost

In [ ]:
S = json.load(open(settings.RESULTS / 'upsampler_scene_metrics.json'))
cost = S['cost']
n_rooms = len(os.listdir(f'{D}/Area_5'))
print('upsampler:', f"{cost['params']:,} parameters | {cost['sec_per_room']:.1f} s/room | peak GPU memory {cost['peak_gpu_mb']:.0f} MB")
tab = []
for lab in ['A_dense_on_dense', 'C_dense_model_on_sparse25', 'E_dense_model_on_interp25x4', 'G_dense_model_on_upsampled25x4', 'H_trained_on_upsampled']:
    f = settings.RESULTS / f'{lab}.json'
    if f.exists():
        r = json.load(open(f))
        up = cost['sec_per_room'] * n_rooms if lab.startswith(('G', 'H')) else 0.0
        tab.append(dict(row=lab, mIoU=round(100 * r['mIoU'], 2), ptv3_test_s=round(r['ptv3_test_seconds'] or 0), upsampling_s=round(up), total_s=round((r['ptv3_test_seconds'] or 0) + up)))
print(pd.DataFrame(tab).to_markdown(index=False))
print(f'\nTraining time per run: see {settings.EXP}/<run>/train.log (total elapsed = sum of the per-epoch times).')

## 7. Qualitative results

In [ ]:
from scipy.spatial import cKDTree
from pcseg import viz
room = next(r for r in sorted(os.listdir(f'{D}/Area_5')) if r.startswith('office'))
dn = load_room(f'{D}/Area_5/{room}', ('coord', 'color', 'segment'))
sp = load_room(f'{S25}/Area_5/{room}', ('coord', 'color'))
up = load_room(f'{UP}/Area_5/{room}', ('coord', 'color'))
pred_up = np.load(settings.EXP / 'A_dense' / 'test_up25x4' / 'result' / f'Area_5-{room}_pred.npy')
_, nn = cKDTree(up['coord']).query(dn['coord'], k=1, workers=-1)
pred = pred_up[nn]
out = viz.plot_panels([('dense (original)', dn['coord'], dn['color']), ('sparse 25%', sp['coord'], sp['color']), ('upsampled x4', up['coord'], up['color']),
                       ('PTv3 prediction', dn['coord'], viz.label_colors(pred)), ('ground truth', dn['coord'], viz.label_colors(dn['segment'])),
                       ('errors (red)', dn['coord'], viz.error_colors(pred, dn['segment']))], settings.FIGURES / f'{room}_pipeline.png', figsize=3.6)
from IPython.display import Image
Image(out)